In [31]:
import os
from mmsdk import mmdatasdk as md

In [32]:
# 步骤一：下载数据集，DATA_PATH是将下载的csd文件存放的目录

DATASET = md.cmu_mosi
DATA_PATH = './cmumosi/'
try:
    md.mmdataset(DATASET.highlevel, DATA_PATH)
    md.mmdataset(DATASET.raw, DATA_PATH)
    md.mmdataset(DATASET.labels, DATA_PATH)
except:
    print('Dataset already exists')

[2025-11-02 10:03:53.322] | Error   | ./data/CMU_MOSI_TimestampedWordVectors.csd file already exists ...
Dataset already exists


In [33]:
# 步骤二：加载csd文件并对齐

data_files = os.listdir(DATA_PATH)
print('\n'.join(data_files))
 


CMU_MOSI_COVAREP.csd
CMU_MOSI_OpenSmile_EB10.csd
CMU_MOSI_openSMILE_IS09.csd
CMU_MOSI_Opinion_Labels.csd
CMU_MOSI_TimestampedPhones.csd
CMU_MOSI_TimestampedWords.csd
CMU_MOSI_TimestampedWordVectors.csd
CMU_MOSI_TimestampedWordVectors_1.1.csd
CMU_MOSI_Visual_Facet_41.csd
CMU_MOSI_Visual_Facet_42.csd
CMU_MOSI_Visual_OpenFace_1.csd
CMU_MOSI_Visual_OpenFace_2.csd


In [36]:
visual_field = 'CMU_MOSI_Visual_Facet_41'
acoustic_field = 'CMU_MOSI_COVAREP'
text_field = 'CMU_MOSI_TimestampedWordVectors'
 
features = [
    text_field,
    visual_field,
    acoustic_field
]
 
recipe = {feat: os.path.join(DATA_PATH, feat) + '.csd' for feat in features}
dataset = md.mmdataset(recipe)
print(list(dataset[text_field].keys())[55])
print(dataset)
print('done!')

[2025-11-02 10:05:12.458] | Success | Computational sequence read from file ./data/CMU_MOSI_TimestampedWordVectors.csd ...
[2025-11-02 10:05:12.473] | Status  | Checking the integrity of the <glove_vectors> computational sequence ...
[2025-11-02 10:05:12.473] | Status  | Checking the format of the data in <glove_vectors> computational sequence ...


  0%|          | 0/93 [00:00<?, ? Computational Sequence Entries/s]

[2025-11-02 10:05:12.589] | Success | <glove_vectors> computational sequence data in correct format.
[2025-11-02 10:05:12.590] | Status  | Checking the format of the metadata in <glove_vectors> computational sequence ...
[2025-11-02 10:05:12.590] | Warning | <glove_vectors> computational sequence does not have all the required metadata ... continuing 
[2025-11-02 10:05:12.591] | Success | Computational sequence read from file ./data/CMU_MOSI_Visual_Facet_41.csd ...
[2025-11-02 10:05:12.608] | Status  | Checking the integrity of the <FACET_4.1> computational sequence ...
[2025-11-02 10:05:12.608] | Status  | Checking the format of the data in <FACET_4.1> computational sequence ...


[2025-11-02 10:05:12.735] | Success | <FACET_4.1> computational sequence data in correct format.
[2025-11-02 10:05:12.735] | Status  | Checking the format of the metadata in <FACET_4.1> computational sequence ...
[2025-11-02 10:05:12.735] | Warning | <FACET_4.1> computational sequence does not have all the required metadata ... continuing 
[2025-11-02 10:05:12.739] | Success | Computational sequence read from file ./data/CMU_MOSI_COVAREP.csd ...
[2025-11-02 10:05:12.757] | Status  | Checking the integrity of the <COVAREP> computational sequence ...
[2025-11-02 10:05:12.757] | Status  | Checking the format of the data in <COVAREP> computational sequence ...


[2025-11-02 10:05:12.852] | Success | <COVAREP> computational sequence data in correct format.
[2025-11-02 10:05:12.852] | Status  | Checking the format of the metadata in <COVAREP> computational sequence ...
[2025-11-02 10:05:12.852] | Warning | <COVAREP> computational sequence does not have all the required metadata ... continuing 
[2025-11-02 10:05:12.852] | Success | Dataset initialized successfully ... 
ZUXBRvtny7o
done!


步骤3：对齐csd文件并保存对齐之后的结果。保存的csd文件会存储到'./deployed'文件夹中，下一次使用数据时直接从deployed文件中加载数据就ok，这样就不用重复下载和对齐。

In [ ]:
import numpy as np
def avg(intervals: np.array, features: np.array) -> np.array:
    try:
        return np.average(features, axis=0)
    except:
        return features
 
# first we align to words with averaging, collapse_function receives a list of functions
dataset.align(text_field, collapse_functions=[avg])
label_field = 'CMU_MOSEI_Opinion_Labels'
 
# we add and align to lables to obtain labeled segments
# this time we don't apply collapse functions so that the temporal sequences are preserved
label_recipe = {label_field: os.path.join(DATA_PATH, label_field + '.csd')}
dataset.add_computational_sequences(label_recipe, destination=None)
dataset.align(label_field, replace=True)
print(list(dataset[text_field].keys())[55])
print('done!')
 
 
###保存
deploy_files={x:x for x in dataset.computational_sequences.keys()}
dataset.deploy("./deployed", deploy_files),
aligned_cmumosi_highlevel=md.mmdataset('./deployed')

In [5]:
# 读取音频部分
import h5py
acoustic_field = 'data\CMU_MOSI_COVAREP.csd'
lan = h5py.File(acoustic_field)
print(lan.keys())
print(lan['COVAREP'].keys())
print(lan['COVAREP']['data'].keys())
print(lan['COVAREP']['metadata'].keys())
print(lan['COVAREP']['data']['03bSnISJMiM'].keys())
print(lan['COVAREP']['data']['03bSnISJMiM']['features'])
print(lan['COVAREP']['data']['03bSnISJMiM']['intervals'])


<KeysViewHDF5 ['COVAREP']>
<KeysViewHDF5 ['data', 'metadata']>
<KeysViewHDF5 ['03bSnISJMiM', '0h-zjBukYpk', '1DmNV9C1hbY', '1iG0909rllw', '2WGyTLYerpo', '2iD-tVS8NPw', '5W7Z1C_fDaE', '6Egk_28TtTM', '6_0THN4chvY', '73jzhE8R1TQ', '7JsX8y1ysxY', '8OtFthrtaJM', '8d-gEyoeBzc', '8qrpnFRGt2A', '9J25DZhivz8', '9T9Hf74oK10', '9c67fiY0wGQ', '9qR7uwkblbs', 'Af8D0E4ZXaw', 'BI97DNYfe5I', 'BXuRRbG0Ugk', 'Bfr499ggo-0', 'BioHAh1qJAQ', 'BvYR0L6f2Ig', 'Ci-AH39fi3Y', 'Clx4VXItLTE', 'Dg_0XKD0Mf4', 'G-xst2euQUc', 'G6GlGvlkxAQ', 'GWuJjcEuzt8', 'HEsqda8_d0Q', 'I5y0__X72p0', 'Iu2PFX3z_1s', 'IumbAb8q2dM', 'Jkswaaud0hk', 'LSi-o-IrDMs', 'MLal-t_vJPM', 'Njd1F0vZSm4', 'Nzq88NnDkEk', 'OQvJTdtJ2H4', 'OtBXNcAL_lE', 'Oz06ZWiO20M', 'POKffnXeBds', 'PZ-lDQFboO8', 'QN9ZIUWUXsY', 'Qr1Ca94K55A', 'Sqr0AcuoNnk', 'TvyZBvOMOTc', 'VCslbP0mgZI', 'VbQk4H8hgr0', 'Vj1wYRQjB-o', 'W8NXH0Djyww', 'WKA5OygbEKI', 'X3j2zQgwYgE', 'ZAIRrfG22O0', 'ZUXBRvtny7o', '_dI--eQ6qVU', 'aiEXnCPZubE', 'atnd_PF-Lbs', 'bOL9jKpeJRs', 'bvLlb-M3UXU', 'c5xsKM

In [4]:
# 读取视频部分
import h5py
visual_field = 'data\CMU_MOSI_Visual_Facet_41.csd'
lan = h5py.File(visual_field)
print(lan.keys())
print(lan['FACET_4.1'].keys())
print(lan['FACET_4.1']['data'].keys())
print(lan['FACET_4.1']['metadata'].keys())
print(lan['FACET_4.1']['data']['03bSnISJMiM'].keys())
print(lan['FACET_4.1']['data']['03bSnISJMiM']['features'])
print(lan['FACET_4.1']['data']['03bSnISJMiM']['intervals'])


<KeysViewHDF5 ['FACET_4.1']>
<KeysViewHDF5 ['data', 'metadata']>
<KeysViewHDF5 ['03bSnISJMiM', '0h-zjBukYpk', '1DmNV9C1hbY', '1iG0909rllw', '2WGyTLYerpo', '2iD-tVS8NPw', '5W7Z1C_fDaE', '6Egk_28TtTM', '6_0THN4chvY', '73jzhE8R1TQ', '7JsX8y1ysxY', '8OtFthrtaJM', '8d-gEyoeBzc', '8qrpnFRGt2A', '9J25DZhivz8', '9T9Hf74oK10', '9c67fiY0wGQ', '9qR7uwkblbs', 'Af8D0E4ZXaw', 'BI97DNYfe5I', 'BXuRRbG0Ugk', 'Bfr499ggo-0', 'BioHAh1qJAQ', 'BvYR0L6f2Ig', 'Ci-AH39fi3Y', 'Clx4VXItLTE', 'Dg_0XKD0Mf4', 'G-xst2euQUc', 'G6GlGvlkxAQ', 'GWuJjcEuzt8', 'HEsqda8_d0Q', 'I5y0__X72p0', 'Iu2PFX3z_1s', 'IumbAb8q2dM', 'Jkswaaud0hk', 'LSi-o-IrDMs', 'MLal-t_vJPM', 'Njd1F0vZSm4', 'Nzq88NnDkEk', 'OQvJTdtJ2H4', 'OtBXNcAL_lE', 'Oz06ZWiO20M', 'POKffnXeBds', 'PZ-lDQFboO8', 'QN9ZIUWUXsY', 'Qr1Ca94K55A', 'Sqr0AcuoNnk', 'TvyZBvOMOTc', 'VCslbP0mgZI', 'VbQk4H8hgr0', 'Vj1wYRQjB-o', 'W8NXH0Djyww', 'WKA5OygbEKI', 'X3j2zQgwYgE', 'ZAIRrfG22O0', 'ZUXBRvtny7o', '_dI--eQ6qVU', 'aiEXnCPZubE', 'atnd_PF-Lbs', 'bOL9jKpeJRs', 'bvLlb-M3UXU', 'c5xs

In [8]:
# 读取文本部分
import h5py
text_field = 'data\CMU_MOSI_TimestampedWordVectors_1.1.csd'
lan = h5py.File(text_field)
print(lan.keys())
print(lan['glove_vectors'].keys())
print(lan['glove_vectors']['data'].keys())
print(lan['glove_vectors']['metadata'].keys())
print(lan['glove_vectors']['data']['03bSnISJMiM'].keys())
print(lan['glove_vectors']['data']['03bSnISJMiM']['features'])
print(lan['glove_vectors']['data']['03bSnISJMiM']['intervals'])


<KeysViewHDF5 ['glove_vectors']>
<KeysViewHDF5 ['data', 'metadata']>
<KeysViewHDF5 ['03bSnISJMiM', '0h-zjBukYpk', '1DmNV9C1hbY', '1iG0909rllw', '2WGyTLYerpo', '2iD-tVS8NPw', '5W7Z1C_fDaE', '6Egk_28TtTM', '6_0THN4chvY', '73jzhE8R1TQ', '7JsX8y1ysxY', '8OtFthrtaJM', '8d-gEyoeBzc', '8qrpnFRGt2A', '9J25DZhivz8', '9T9Hf74oK10', '9c67fiY0wGQ', '9qR7uwkblbs', 'Af8D0E4ZXaw', 'BI97DNYfe5I', 'BXuRRbG0Ugk', 'Bfr499ggo-0', 'BioHAh1qJAQ', 'BvYR0L6f2Ig', 'Ci-AH39fi3Y', 'Clx4VXItLTE', 'Dg_0XKD0Mf4', 'G-xst2euQUc', 'G6GlGvlkxAQ', 'GWuJjcEuzt8', 'HEsqda8_d0Q', 'I5y0__X72p0', 'Iu2PFX3z_1s', 'IumbAb8q2dM', 'Jkswaaud0hk', 'LSi-o-IrDMs', 'MLal-t_vJPM', 'Njd1F0vZSm4', 'Nzq88NnDkEk', 'OQvJTdtJ2H4', 'OtBXNcAL_lE', 'Oz06ZWiO20M', 'POKffnXeBds', 'PZ-lDQFboO8', 'QN9ZIUWUXsY', 'Qr1Ca94K55A', 'Sqr0AcuoNnk', 'TvyZBvOMOTc', 'VCslbP0mgZI', 'VbQk4H8hgr0', 'Vj1wYRQjB-o', 'W8NXH0Djyww', 'WKA5OygbEKI', 'X3j2zQgwYgE', 'ZAIRrfG22O0', 'ZUXBRvtny7o', '_dI--eQ6qVU', 'aiEXnCPZubE', 'atnd_PF-Lbs', 'bOL9jKpeJRs', 'bvLlb-M3UXU', '

In [29]:
# 读取标签表
import h5py
label_field = 'data\CMU_MOSI_Opinion_Labels.csd'
lan = h5py.File(label_field)
print(lan.keys())
print(lan['Opinion Segment Labels']['data'])
group = lan['Opinion Segment Labels']['data']['03bSnISJMiM']
# print(group.name, type(group), group.keys(), group.values())
print(group.name)
print(type(group))
print(group.keys())
print(group.values())

for (index, key) in enumerate(group.keys()):
    print(index,key)
    print(len(group[key][()]))
    print(group[key][()].shape)  
    print(group[key][()])
    print(group[key][(12)][0])
    # data.append(group[key][()])  # 把数据拿到数组里来

<KeysViewHDF5 ['Opinion Segment Labels']>
<HDF5 group "/Opinion Segment Labels/data" (93 members)>
/Opinion Segment Labels/data/03bSnISJMiM
<class 'h5py._hl.group.Group'>
<KeysViewHDF5 ['features', 'intervals']>
ValuesViewHDF5(<HDF5 group "/Opinion Segment Labels/data/03bSnISJMiM" (2 members)>)
0 features
13
(13, 1)
[[ 2.4 ]
 [-0.8 ]
 [-1.  ]
 [-1.75]
 [ 0.  ]
 [ 0.  ]
 [ 0.8 ]
 [ 0.  ]
 [ 0.2 ]
 [-1.2 ]
 [-0.5 ]
 [ 2.2 ]
 [ 1.8 ]]
1.8
1 intervals
13
(13, 2)
[[ 51.904533  55.94535 ]
 [ 56.045124  66.78072 ]
 [ 66.78072   68.73628 ]
 [ 68.73628   70.542175]
 [ 70.542175  71.69955 ]
 [ 71.69955   72.85692 ]
 [ 72.85692   77.79569 ]
 [ 77.79569   89.52902 ]
 [ 89.52902   92.23288 ]
 [ 92.23288   94.80703 ]
 [ 94.80703   96.57301 ]
 [ 96.57301   99.01746 ]
 [168.65918  170.24557 ]]
168.65918


In [2]:
# 查看CMU_MOSEI_Labels.csd的具体内容
print("\n=== 查看CMU_MOSEI_Labels.csd的具体内容 ===")
import h5py
import os

# CMU_MOSEI_Labels.csd文件路径
mosei_labels_path = os.path.join('cmumosei', 'CMU_MOSEI_Labels.csd')

# 检查文件是否存在
if os.path.exists(mosei_labels_path):
    # 打开并读取文件内容
    with h5py.File(mosei_labels_path, 'r') as f:
        print("文件顶层键：", list(f.keys()))
        
        # 遍历所有标签类型
        for label_key in f.keys():
            print(f"\n--- {label_key} ---\n")
            print(f"{label_key}的键：", list(f[label_key].keys()))
            
            # 查看数据键
            if 'data' in f[label_key]:
                print("数据样本键数量：", len(list(f[label_key]['data'].keys())))
                print("前5个数据样本键：", list(f[label_key]['data'].keys())[:5])
                
                # 查看第一个数据样本的结构
                first_sample_key = list(f[label_key]['data'].keys())[0]
                print(f"\n第一个样本({first_sample_key})的结构：", list(f[label_key]['data'][first_sample_key].keys()))
                
                # 查看features和intervals的形状
                if 'features' in f[label_key]['data'][first_sample_key]:
                    print(f"features形状：", f[label_key]['data'][first_sample_key]['features'].shape)
                    print(f"features前5个值：")
                    print(f[label_key]['data'][first_sample_key]['features'][:5])
                
                if 'intervals' in f[label_key]['data'][first_sample_key]:
                    print(f"intervals形状：", f[label_key]['data'][first_sample_key]['intervals'].shape)
                    print(f"intervals前5个值：")
                    print(f[label_key]['data'][first_sample_key]['intervals'][:5])
            
            # 查看元数据
            if 'metadata' in f[label_key]:
                print("\n元数据键：", list(f[label_key]['metadata'].keys()))
                for meta_key in f[label_key]['metadata']:
                    try:
                        print(f"{meta_key}: {f[label_key]['metadata'][meta_key][()]}")
                    except:
                        print(f"{meta_key}: [无法直接打印的复杂数据]")
else:
    print(f"文件不存在: {mosei_labels_path}")
    print("请确认文件路径是否正确")
    
    # 尝试其他可能的路径
    alternative_path = 'cmumosei/labels/CMU_MOSEI_Labels.csd'
    print(f"\n尝试替代路径: {alternative_path}")
    if os.path.exists(alternative_path):
        with h5py.File(alternative_path, 'r') as f:
            print("文件顶层键：", list(f.keys()))



=== 查看CMU_MOSEI_Labels.csd的具体内容 ===
文件顶层键： ['All Labels']

--- All Labels ---

All Labels的键： ['data', 'metadata']
数据样本键数量： 3293
前5个数据样本键： ['--qXJuDtHPw', '-3g5yACwYnA', '-3nNcZdcdvU', '-571d8cVauQ', '-6rXp3zJ3kc']

第一个样本(--qXJuDtHPw)的结构： ['features', 'intervals']
features形状： (1, 7)
features前5个值：
[[1.        0.6666667 0.        0.        0.        0.        0.       ]]
intervals形状： (1, 2)
intervals前5个值：
[[23.199 30.325]]

元数据键： ['alignment compatible', 'computational sequence description', 'computational sequence version', 'contact', 'creator', 'dataset bib citation', 'dataset name', 'dataset version', 'dimension names', 'featureset bib citation', 'md5', 'root name', 'uuid']
alignment compatible: [b'true']
computational sequence description: [b'"Labels for CMU-MOSEI Dataset"']
computational sequence version: [b'1.0']
contact: [b'"abagherz@andrew.cmu.edu"']
creator: [b'"Amir Zadeh"']
dataset bib citation: [b'"@inproceedings{cmumoseiacl2018, title={Multimodal Language Analysis in the Wil